In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats


## Load the Three-Layer Synthetic Seismic Data

In [ ]:
# Load the three-layer model data
data = np.load('long_offset_three_layer.npz', allow_pickle=True)

rx   = data['rx']          # receiver x positions (m)
rz   = data['rz']          # receiver z positions (m)
sx   = float(data['sx'])   # source x position (m)
sz   = float(data['sz'])   # source z position (m)
dt   = float(data['dt'])   # time step (ms)
seismic = data['data']     # seismic data: shape (n_samples, n_traces)

n_samples, n_traces = seismic.shape
t_axis = np.arange(n_samples) * dt   # time axis in ms
offsets = np.abs(rx - sx)             # source-receiver offsets (m)

print(f"Loaded: {n_traces} traces, {n_samples} time samples, dt={dt:.4f} ms")
print(f"Offset range: {offsets.min():.0f} – {offsets.max():.0f} m")
print(f"Total record length: {t_axis[-1]:.1f} ms")


## Wiggle Plot of the Shot Gather

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
scale = 200   # display scaling factor

good_idx = np.where(rx >= sx)[0]
for i in good_idx:
    offset = offsets[i]
    trace  = seismic[:, i]
    trace_norm = trace / (np.max(np.abs(trace)) + 1e-10)
    ax.plot(t_axis, offset + scale * trace_norm, 'k-', lw=0.4)

ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Offset (m)', fontsize=12)
ax.set_title('Three-Layer Model — Seismic Shot Gather', fontsize=13)
ax.set_xlim(0, 6000)
plt.tight_layout()
plt.show()


## First-Break Travel Time Picks

First-break times were picked by identifying the first sample on each trace
that exceeds 1% of that trace's maximum amplitude.  The picks are stored as
(offset, travel-time) pairs for the refraction analysis.

In [ ]:
# Auto-pick first breaks using 1% amplitude threshold on each trace
picks_offset = []
picks_time_ms = []

for i in np.where(rx > sx)[0]:
    trace   = seismic[:, i]
    thresh  = 0.01 * np.max(np.abs(trace))
    above   = np.where(np.abs(trace) > thresh)[0]
    if len(above) > 0:
        picks_offset.append(offsets[i])
        picks_time_ms.append(t_axis[above[0]])

picks_offset   = np.array(picks_offset)
picks_time_ms  = np.array(picks_time_ms)
picks_time_s   = picks_time_ms / 1000.0

print(f"Picked {len(picks_offset)} first-break times")
print(f"{'Offset (m)':>12}  {'Travel Time (ms)':>18}")
for x, t in zip(picks_offset, picks_time_ms):
    print(f"{x:>12.0f}  {t:>18.2f}")


## T-X Plot and Linear Regression

Three linear segments are visible in the T-X plot corresponding to:
- **Segment 1** (direct wave through Layer 1): offsets 200 – 1600 m
- **Segment 2** (head wave refracted at Layer 1/2 boundary): offsets 2000 – 4000 m
- **Segment 3** (head wave refracted at Layer 2/3 boundary): offsets 5600 – 9400 m

In [ ]:
# Define segment masks
seg1 = (picks_offset <= 1600)
seg2 = (picks_offset >= 2000) & (picks_offset <= 4000)
seg3 = (picks_offset >= 5600)

segment_labels  = ['Segment 1 (V₁)', 'Segment 2 (V₂)', 'Segment 3 (V₃)']
segment_colors  = ['royalblue', 'seagreen', 'tomato']
segment_masks   = [seg1, seg2, seg3]

results = {}
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(picks_offset, picks_time_s, color='black', s=25, zorder=5, label='First-break picks')

for mask, label, color in zip(segment_masks, segment_labels, segment_colors):
    x = picks_offset[mask]
    t = picks_time_s[mask]
    slope, intercept, r, _, _ = stats.linregress(x, t)
    velocity = 1.0 / slope
    results[label] = {'slope': slope, 'intercept': intercept, 'V': velocity, 'R2': r**2}
    x_line = np.array([0, 10000])
    ax.plot(x_line, slope * x_line + intercept, color=color, lw=2,
            label=f"{label}: V = {velocity:.0f} m/s")

ax.set_xlabel('Offset (m)', fontsize=12)
ax.set_ylabel('Travel Time (s)', fontsize=12)
ax.set_title('T-X Plot — Three-Layer Refraction Analysis', fontsize=13)
ax.set_xlim(0, 10000)
ax.set_ylim(0, 5.5)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("
Linear Regression Results:")
for label, res in results.items():
    print(f"  {label}: V = {res['V']:.1f} m/s,  intercept = {res['intercept']*1000:.2f} ms,  R² = {res['R2']:.5f}")


## Layer Depth Calculation — Intercept-Time Method

Using the intercept-time method:

**Depth to Layer 2 (h₁):**

59h_1 = \frac{t_{i2}}{2} \cdot \frac{V_1 V_2}{\sqrt{V_2^2 - V_1^2}}59

**Depth to Layer 3 (h₁ + h₂):**

59t_{i3} = \frac{2h_1 \cos(i_{c1,3})}{V_1} + \frac{2h_2 \cos(i_{c2,3})}{V_2}59

where {c1,3} = \arcsin(V_1/V_3)$ and {c2,3} = \arcsin(V_2/V_3)$

In [ ]:
V1  = results['Segment 1 (V₁)']['V']
V2  = results['Segment 2 (V₂)']['V']
V3  = results['Segment 3 (V₃)']['V']
ti2 = results['Segment 2 (V₂)']['intercept']   # seconds
ti3 = results['Segment 3 (V₃)']['intercept']   # seconds

# Depth to top of Layer 2
h1 = (ti2 / 2.0) * (V1 * V2) / np.sqrt(V2**2 - V1**2)

# Critical angles for Layer 3 refractor
ic1_3 = np.arcsin(V1 / V3)
ic2_3 = np.arcsin(V2 / V3)

# Thickness of Layer 2
h2 = (ti3 - 2.0 * h1 * np.cos(ic1_3) / V1) / (2.0 * np.cos(ic2_3) / V2)

print("=" * 45)
print("   REFRACTION ANALYSIS — SUMMARY TABLE")
print("=" * 45)
print(f"  V1  (Layer 1 velocity) :  {V1:8.1f}  m/s")
print(f"  V2  (Layer 2 velocity) :  {V2:8.1f}  m/s")
print(f"  V3  (Layer 3 velocity) :  {V3:8.1f}  m/s")
print("-" * 45)
print(f"  t_i2 (intercept time)  :  {ti2*1000:8.2f}  ms")
print(f"  t_i3 (intercept time)  :  {ti3*1000:8.2f}  ms")
print("-" * 45)
print(f"  h1   (depth to Layer 2):  {h1:8.1f}  m")
print(f"  h2   (Layer 2 thickness): {h2:8.1f}  m")
print(f"  h1+h2 (depth to Layer 3): {h1+h2:8.1f}  m")
print("=" * 45)
